# Spillovers or common shocks? Growth across 30 European economies, 1996–2025

**Real data, 30 countries over 30 years.** Notebook `08` showed on simulated
data that a spatial panel estimator cannot tell a spillover between neighbours
from a common shock that hits everyone, unless the model is told about the
shock. This notebook runs the same comparison on the EU27, Norway, Switzerland
and Iceland, using the European Commission's AMECO database, where nobody
knows the right answer.

The model is a growth regression in panel form: annual real GDP growth on the
investment share and population growth, with a spatial lag of growth over the
four nearest countries. The question is how large the spatial lag is.

The answer depends on what else is in the model:

| | ρ̂ |
|---|---|
| country effects | 0.70 |
| country and year effects | 0.48 |
| spatial CCE (common factors) | −0.04, standard error 0.24 |

and on how many neighbours each country gets, from 0.36 with two to 0.69 with
eight. Over all of that, **the investment coefficient stays between 0.18 and
0.22.** That contrast is the main result.

Thirty units is small. It is enough for a static panel with 30 years, not for
any method whose asymptotics need many units. AMECO's forecast years are
already removed from the data; the last year is 2025. Data and licensing are
in `data/europe_PROVENANCE.md`.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Printf, Statistics, LinearAlgebra, SparseArrays
import StatsAPI: residuals

const DATA = joinpath(@__DIR__, "data")

## 1. A balanced panel

In [ ]:
function readcsv(path)
    lines = readlines(path)
    hdr = split(lines[1], ',')
    rows = [split(l, ','; limit = length(hdr)) for l in lines[2:end] if !isempty(l)]
    return hdr, rows
end

ah, arows = readcsv(joinpath(DATA, "europe_ameco.csv"))
ch, crows = readcsv(joinpath(DATA, "europe_countries.csv"))

ameco = Dict((r[1], parse(Int, r[3])) => r for r in arows)
function val(id, year, column)
    r = get(ameco, (id, year), nothing)
    j = findfirst(==(column), ah)
    (r === nothing || isempty(r[j])) ? NaN : parse(Float64, r[j])
end

ids  = [r[1] for r in crows]
name = [strip(r[end], '"') for r in crows]
xy   = hcat(parse.(Float64, [r[3] for r in crows]), parse.(Float64, [r[4] for r in crows])) ./ 1000
N    = length(ids)

const YEARS = 1996:2025
T = length(YEARS)

# N x T matrices; vec() stacks them period by period, the layout the panel fitters expect.
growth = [100 * log(val(c, t, "real_gdp_bn_nc") / val(c, t - 1, "real_gdp_bn_nc")) for c in ids, t in YEARS]
invest = [100 * val(c, t, "gfcf_bn_eur") / val(c, t, "gdp_bn_eur") for c in ids, t in YEARS]
popg   = [100 * log(val(c, t, "population_ths") / val(c, t - 1, "population_ths")) for c in ids, t in YEARS]

@printf("%d countries x %d years = %d observations; missing values: %d\n",
        N, T, N * T, count(isnan, growth) + count(isnan, invest) + count(isnan, popg))
println(join(ids, " "))

No gaps: every country has every variable for every year, so the panel is
balanced. Growth is the log change in real GDP in national currency, so
exchange rates do not enter. The investment share is gross fixed capital
formation over GDP, both in current euros.

## 2. The common shocks

Before any model, look at the cross-sectional mean of growth, year by year.

In [ ]:
cross = vec(mean(growth; dims = 1))
println("year   mean growth   countries with negative growth")
for (k, t) in enumerate(YEARS)
    @printf("%d   %+6.2f        %2d %s\n", t, cross[k], count(<(0), growth[:, k]),
            cross[k] < 0 ? "<--" : "")
end

Three years stand out. In **2009, 29 of 30 economies shrank**; Poland was the
exception. In **2020, 28 shrank**; Ireland grew 6.9%, on multinational
profits again, and Lithuania was flat. In 2012, at the bottom of the euro crisis,
14 did. Those are shocks that hit nearly everyone at once, whether or not they
share a border. A spatial lag over neighbours will find them, because a
neighbour's growth is a good predictor of yours in any year when everyone's
growth moves together.

In [ ]:
y = vec(growth)
X = hcat(vec(invest), vec(popg))
vn = ["investment_share", "population_growth"]

cd = pesaran_cd(y, N, T)
ae = alpha_exponent(y, N, T)
@printf("Pesaran CD on growth: %.2f (p = %.2g)\n", cd.stat, cd.p_value)
@printf("exponent of cross-sectional dependence: alpha = %.3f  [%.2f, %.2f]\n",
        ae.alpha, ae.ci[1], ae.ci[2])

The statistics agree. Pesaran's CD is **81.7**, and the exponent of
cross-sectional dependence is **α̂ = 0.92**, with an interval from 0.84 to
0.99. An α of 1 is a factor that loads on every unit; values above 1/2 are
strong dependence that no spatial weights matrix with a bounded number of
neighbours can produce. Something common is moving these 30 economies, and it
has to be dealt with before ρ̂ means anything.

## 3. Weights for 30 countries

In [ ]:
Wq, qid = read_gal(joinpath(DATA, "europe_countries_queen.gal"))
@printf("GAL ids match the country file: %s\n", qid == ids)
println("no land neighbour: ", join(ids[vec(sum(Wq; dims = 2)) .== 0], " "))

W4 = knn_weights(xy, 4)
for c in ("IE", "CY", "MT", "IS", "PT", "FI")
    i = findfirst(==(c), ids)
    println(rpad(c, 4), join(ids[findall(>(0), W4[i, :])], " "))
end

Contiguity leaves four countries without a neighbour: Cyprus, Ireland,
Iceland and Malta. Ireland's only land border is with the United Kingdom,
which is not in the data. Four nearest neighbours by distance between
centroids gives everyone neighbours, and the lists are sensible, if not
obvious: Ireland's are the Benelux and France, Iceland's are Scandinavia and
Ireland. Section 5 shows how much the choice of k matters. The answer is a
lot.

## 4. Three ways to handle the common shocks

**Country effects only.** Each country has its own mean growth rate, and the
common shocks are left in the error term.

**Country and year effects.** The year effects remove the cross-sectional
mean of each year: a shock that hits every country *equally*.

In [ ]:
fit(W, fe) = fit_spatial_panel(SAR(), fe, W, X, y, N, T;
                               approach = DirectQML(), method = ExactLU(), varnames = vn)
m_ind = fit(W4, IndivFE())
m_two = fit(W4, TwoWayFE())
println(m_ind)
println(m_two)

**ρ̂ falls from 0.70 to 0.48** when the year effects are added. About a third
of what looked like spillovers between neighbours was the common shock. The
investment coefficient moves from 0.178 to 0.197. Population growth is not
significant in either model.

In [ ]:
for (lab, m) in (("country effects", m_ind), ("country and year effects", m_two))
    r  = residuals(m)
    c2 = pesaran_cd(r, N, T)
    @printf("%-26s residual CD %+7.2f (p = %.2g)   ", lab, c2.stat, c2.p_value)
    try
        a2 = alpha_exponent(r, N, T)
        @printf("alpha %.3f [%.2f, %.2f]\n", a2.alpha, a2.ci[1], a2.ci[2])
    catch err
        println("alpha not identified (at or below 1/2)")
    end
end

The residual diagnostics show why the year effects were needed and what they
leave behind. With country effects only, the residuals still have CD = 11.1
and α̂ = 0.66: strong dependence, so the common shocks are in the error term
and ρ̂ = 0.70 is biased. With year effects, CD falls to −2.35 and α̂ is no
longer identified, which is the package's way of saying that the remaining
dependence is weak.

That is not the same as no common factor. Year effects remove a shock that
hits every country *equally*. A shock that hits Greece harder than Germany,
which is what the euro crisis was, leaves a residual that two-way effects
cannot remove. Spatial CCE handles that case: it projects out the
cross-sectional means of y and X with a separate loading for each country.

In [ ]:
m_cce = fit_sp_cce(y, X, W4, N, T; coefnames = vn)
println(m_cce)

**Under CCE, ρ̂ is −0.04 with a standard error of 0.24.** There is no
detectable spillover once common factors with heterogeneous loadings are
removed. The investment coefficient is 0.195, the same as under two-way
effects.

Before you report that, read the last line of the output. **The Sargan test
rejects the instruments, p = 0.012.** CCE instruments Wy with WX and W²X,
and the test says those instruments are not valid here. The first stage
is strong enough (F = 13.7), so the problem is validity, not weakness.
CCE's ρ̂ is not a clean answer either.

In [ ]:
@printf("%-34s %8s %8s %12s\n", "", "rho", "se", "investment")
@printf("%-34s %8.3f %8.3f %12.3f\n", "SAR, country effects", m_ind.rho, m_ind.rho_se, m_ind.coef[1])
@printf("%-34s %8.3f %8.3f %12.3f\n", "SAR, country and year effects", m_two.rho, m_two.rho_se, m_two.coef[1])
@printf("%-34s %8.3f %8.3f %12.3f\n", "spatial CCE", m_cce.rho, sqrt(m_cce.vcov[1, 1]), m_cce.coef[1])

So three estimators give three answers for ρ, from −0.04 to 0.70, and the one
that deals best with heterogeneous common shocks fails its own specification
test. The honest conclusion is that **these data do not identify a spillover
between European economies separately from common shocks.** That is the same
conclusion as notebook `08`, reached here without knowing the truth.

The investment coefficient is a different story. It is 0.18–0.20 under all
three. That stability is what a result that does not depend on the
treatment of cross-sectional dependence looks like.

## 5. ρ̂ depends on the number of neighbours

In [ ]:
@printf("%-24s %8s %8s %12s\n", "weights", "rho", "se", "investment")
for k in (2, 3, 4, 5, 6, 8)
    m = fit(knn_weights(xy, k), TwoWayFE())
    @printf("%-24s %8.3f %8.3f %12.3f\n", "$k nearest", m.rho, m.rho_se, m.coef[1])
end

**ρ̂ rises with every neighbour added: from 0.36 with two to 0.69 with
eight.** This is with year effects in, so the equal-loading shock is already
removed. The investment coefficient moves from 0.190 to 0.200.

A spillover parameter that doubles when the definition of "neighbour" changes
is not a property of the economies. With eight neighbours out of 29, the
spatial lag is close to a regional average of Europe, and a regional average
absorbs the shocks that hit one part of Europe harder than the rest. Choose k
before looking at ρ̂, and report how ρ̂ changes with k.

In [ ]:
keep = findall(>(0), vec(sum(Wq; dims = 2)))
Nk   = length(keep)
Wk   = row_standardize(sparse(Float64.(Matrix(Wq)[keep, keep] .> 0)))
rows_k = vcat([((t - 1) * N) .+ keep for t in 1:T]...)
mq = fit_spatial_panel(SAR(), TwoWayFE(), Wk, X[rows_k, :], y[rows_k], Nk, T;
                       approach = DirectQML(), method = ExactLU(), varnames = vn)
@printf("queen contiguity, %d countries: rho %.3f (se %.3f), investment %.3f\n",
        Nk, mq.rho, mq.rho_se, mq.coef[1])

Queen contiguity, with the four countries that have no land neighbour left
out, gives ρ̂ = 0.41. That is between the two- and three-neighbour results,
which fits: most countries have two to four land neighbours.

## 6. Standard errors robust to common shocks

Driscoll–Kraay standard errors are robust to any form of cross-sectional
dependence, including common factors, so they are the natural check here.

In [ ]:
println(robust_coeftable(m_two, DriscollKraayVCOV(max_lag = 3, kernel = BartlettKernel())))

The investment coefficient's standard error doubles, from 0.026 to 0.052, and
its z is still 3.8. It survives.

ρ̂'s standard error *falls*, from 0.042 to 0.015. Do not read that as greater
precision. The same thing happened with Conley errors in notebooks `11` and
`12`. A sandwich estimator around the parameter that already models the
cross-sectional dependence does not measure its precision well, and with
N = 30 there are few cross-sections to average over.

## What this notebook is for

- **Measure the common factor before you estimate a spillover.** Here CD was
  82 and α̂ was 0.92. A spatial lag fitted without year effects absorbed about
  a third of that as "spillover".
- **Year effects remove a shock that hits everyone equally; CCE removes one
  that does not.** They gave ρ̂ = 0.48 and −0.04.
- **Read the specification test that comes with the estimator.** CCE's Sargan
  test rejected, so its ρ̂ = −0.04 is not the answer either.
- **Report ρ̂ over a range of k.** It doubled between two and eight
  neighbours.
- **Look for the coefficient that does not move.** The investment
  coefficient was 0.18–0.22 under every treatment of dependence, every k and
  Driscoll–Kraay errors.

## Exercises

1. Fit CCE with `mean_group = true`, which lets each country have its own
   slopes. Does the investment coefficient survive heterogeneous slopes, or
   was its stability a result of pooling?
2. Drop 2009 and 2020 and refit the two-way model. If ρ̂ falls a lot, what
   does that say about whether it measures spillovers?
3. Replace the four nearest neighbours with trade weights: the share of each
   country's exports that go to each partner. Trade is the channel through
   which a spillover would travel. What would it take to build that matrix
   from public data, and why might its ρ̂ still be confounded with common
   shocks?
4. Split the sample at 2008 and refit each half with two-way effects. Is ρ̂
   higher after the euro crisis? Before you run it, say what result you would
   take as evidence of contagion.
5. Ireland's GDP includes the profits of multinationals (see notebook `12`).
   Leave Ireland out. Which of the results in this notebook move?